# Week 5 Exercise: Company Website RAG Assistant (cwait)

A **private knowledge worker** for the Insurellm company website:

1. **Assemble** all company knowledge base files (about, overview, culture, careers) in one place  
2. **Vectorise** everything in **ChromaDB**  
3. **Conversation AI** — ask questions via a Gradio chat UI  

*Run all cells in order. Use the last cell to launch the Gradio chat.*

## 1. Setup & paths

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

load_dotenv(override=True)

# Paths: support running from repo root or from cwait directory
NB_DIR = Path.cwd()
if (NB_DIR / "week5" / "knowledge-base" / "company").exists():
    COMPANY_KB_PATH = (NB_DIR / "week5" / "knowledge-base" / "company").resolve()
elif (NB_DIR / "knowledge-base" / "company").exists():
    COMPANY_KB_PATH = (NB_DIR / "knowledge-base" / "company").resolve()
else:
    COMPANY_KB_PATH = (NB_DIR / ".." / ".." / "knowledge-base" / "company").resolve()
DB_NAME = "company_chroma_db"
PERSIST_DIR = NB_DIR / DB_NAME

print("Company KB:", COMPANY_KB_PATH)
print("Exists:", COMPANY_KB_PATH.is_dir())
print("Chroma persist:", PERSIST_DIR)

## 2. Config

In [ ]:
MODEL = "gpt-4.1-nano"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
K_RETRIEVE = 4
REBUILD_DB = False  # Set True to re-ingest and rebuild Chroma

## 3. Load company documents & chunk

In [ ]:
loader = DirectoryLoader(
    str(COMPANY_KB_PATH),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()
for doc in documents:
    doc.metadata["doc_type"] = "company"

print(f"Loaded {len(documents)} documents")
for d in documents:
    print(" -", d.metadata.get("source", "?")[-50:])

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

## 4. Vectorise in ChromaDB

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

if REBUILD_DB and PERSIST_DIR.exists():
    Chroma(persist_directory=str(PERSIST_DIR), embedding_function=embeddings).delete_collection()

if PERSIST_DIR.exists() and not REBUILD_DB:
    vectorstore = Chroma(persist_directory=str(PERSIST_DIR), embedding_function=embeddings)
    print("Loaded existing Chroma DB")
else:
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=str(PERSIST_DIR),
    )
    print("Built new Chroma DB")

print("Vector count:", vectorstore._collection.count())

## 5. RAG: retriever + LLM

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """You are a friendly, professional assistant for Insurellm's public website.
You answer questions from the public about the company: who we are, our culture, careers, and overview.

Use ONLY the context below to answer. If the answer is not in the context, say so politely.
Do not make up information. Keep answers concise and suitable for a company website.

Context:
{context}
"""

retriever = vectorstore.as_retriever(search_kwargs={"k": K_RETRIEVE})
llm = ChatOpenAI(temperature=0, model=MODEL)

def answer_question(question: str, history):
    if not question or not question.strip():
        return ""
    docs = retriever.invoke(question.strip())
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ])
    return response.content

## 6. Gradio chat UI

In [ ]:
gr.ChatInterface(
    answer_question,
    title="Insurellm Company Q&A",
    description="Ask about Insurellm: company overview, culture, careers, and more.",
).launch(inbrowser=True)